# Lab 2 · One defensible feature

**Pair (group G1):** **Baptiste Vial** · **Vicente Rodriguez**

**Due:** Friday 9 October 2026, 11:10 (Santiago) · **Notebook:** `AIW_Lab02_G1_one_feature.ipynb` · **Data:** `w04_race_weekends_synthetic_v3.csv` (same folder)

**Question.** Can one feature built only from information available *before the race* improve validation balanced accuracy for the top-10 prediction?

**Prediction moment (from Week 4, Section 1):** after qualifying, before the race. Only columns known at that moment may be inputs; `top10` is the race outcome and is never an input.

**How this notebook is organised.** This is our pair's copy of `W04_Thu_Pipelines_Student_v3.ipynb`. Sections 1–8 are the Week 4 cells with their code unchanged, so *Restart & Run All* rebuilds the same split, pipeline and dummy baseline. Our lab work is **L0** (proposal, placed before the code) and **L1–L6** at the end:

| Lab step | Where |
|---|---|
| 2. Proposal | L0 |
| 3. One candidate feature | L1 (`In [9]`) |
| 4. Controlled comparison + three-row score table | L2 (`In [10]`) |
| 5. Decision | L3 diagnostic (`In [11]`) + L4 |
| 6. AI audit, AI-use record, versions, runbook | L5, L6 (`In [12]`) |

**Split (unchanged):** train weeks 1–10 (200 rows) · validation weeks 11–14 (80 rows) · reserved test weeks 15–18 (80 rows). **The reserved test is never scored in this lab.**

*Synthetic teaching data: nothing here says anything about real Formula 1 performance.*

## L0. Proposal 

- **Feature name:** `estimated_grid_position`
- **Formula:** `qualifying_position + grid_penalty`
- **Source columns and when they are known:** `qualifying_position` is known when qualifying ends; `grid_penalty` (grid places lost) is announced before the race starts. Both are known before the race. We do not use `top10`, finishing position or any race result.
- **Expected direction:** no change or a very small **increase** in validation balanced accuracy. A penalty moves a driver back on the starting grid, so the corrected start slot should track top-10 chances better than qualifying alone. However, the model already receives both columns, and logistic regression can learn a weighted sum, so the gain may be zero.

# A professional prediction workflow with `Pipeline`

**Thursday 1 October · 12:30–15:00 · work in pairs**

**Question.** After qualifying, which drivers might finish in the top ten? This is an **invented teaching dataset**, not Formula 1 results. It contains 18 race weekends and 20 fictional drivers per weekend (360 rows). One row represents one driver in one weekend. There are missing numbers, a missing category, and a new circuit category later in time.

**Why this notebook?** In a manual workflow you must remember to fit each imputer, encoder and scaler on the right rows, then apply the same fitted objects in exactly the same order later. A `Pipeline` packages those steps with the model. `fit(X_train, y_train)` learns them together; `predict(X_later)` reuses them. It also lets a parameter search refit the **whole procedure** inside each temporal fold. A pipeline does **not** decide which rows are training or whether a feature is available at prediction time.

**By 15:00, hand in:** one supplied Thursday worksheet per pair and one individual exit ticket. Use short labels in English or Spanish. Predict before running each marked cell. If execution fails, read the saved outputs and explain them.

**Route:** 1 inspect and split → 2 build → 3 fit and inspect → 4 validate → 5 compare temporal folds → 6 apply to new rows → 7 make one controlled change and diagnose limits. Work through the stop points on your worksheet.

## 1. Inspect the information and choose the split

Available **after qualifying, before the race**: qualifying position, practice time gap, previous top-ten rate, grid penalty, circuit type and rain forecast. The target `top10` is the race outcome; it is **never an input feature**. `driver_id` and `week` identify rows and control the split; they are not model features.

**Stop 1.** Predict how many rows belong to training (weeks 1–10), validation (11–14), and the reserved test (15–18). Why must we split before fitting missing-value rules?

In [1]:
from pathlib import Path
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.model_selection import GridSearchCV

path = Path("w04_race_weekends_synthetic_v3.csv")
if not path.exists():
    raise FileNotFoundError("Open the notebook in the folder containing its CSV file.")
data = pd.read_csv(path)
print("Rows, columns:", data.shape)
print(data.head(5).to_string(index=False))
print("The displayed rows are from week 1; later outcomes are not inspected here.")

Rows, columns: (360, 9)
 week driver_id  qualifying_position  practice_gap_s  previous_top10_rate  grid_penalty circuit_type rain_forecast  top10
    1       D01                    4           0.302                0.355             0       street            no      1
    1       D02                    3           0.168                0.448             0       street            no      1
    1       D03                   13           0.981                0.676             0       street            no      0
    1       D04                    6           0.381                0.553             0       street            no      1
    1       D05                    1           0.020                0.630             5       street            no      1
The displayed rows are from week 1; later outcomes are not inspected here.


In [2]:
train = data.loc[data.week <= 10].copy()
valid = data.loc[data.week.between(11, 14)].copy()
test_reserved = data.loc[data.week.between(15, 18)].copy()  # Keep sealed today.
numeric = ["qualifying_position", "practice_gap_s", "previous_top10_rate", "grid_penalty"]
categorical = ["circuit_type", "rain_forecast"]
features = numeric + categorical
X_train, y_train = train[features], train.top10
X_valid, y_valid = valid[features], valid.top10
print("Training / validation / reserved test:", len(train), len(valid), len(test_reserved))
print("Validation circuits:", sorted(valid.circuit_type.unique()))
print("Training circuits:", sorted(train.circuit_type.unique()))
print("Missing training inputs:", train[features].isna().sum().to_dict())
print("Top-ten outcomes in each training weekend:", train.groupby("week").top10.sum().unique().tolist())
assert train.week.max() < valid.week.min() < test_reserved.week.min()
assert "top10" not in features and "week" not in features

Training / validation / reserved test: 200 80 80
Validation circuits: ['high_speed', 'road', 'street', 'technical']
Training circuits: ['high_speed', 'road', 'street']
Missing training inputs: {'qualifying_position': 0, 'practice_gap_s': 17, 'previous_top10_rate': 13, 'grid_penalty': 0, 'circuit_type': 0, 'rain_forecast': 14}
Top-ten outcomes in each training weekend: [10]


## 2. Build one estimator from several steps

The **numeric branch** replaces missing values with the **training median** and standardises each feature using training statistics. The **categorical branch** replaces a missing category with the most frequent training category and one-hot encodes it. `handle_unknown="ignore"` allows a later `technical` circuit even though none appears in training. Both branches feed a logistic regression classifier.

`ColumnTransformer` runs different preprocessing on different columns; the outer `Pipeline` joins preprocessing and model. The names `prep`, `num`, `cat` and `model` let us inspect and tune steps. `__` reaches into a named step, such as `model__C`.

**Stop 2.** Which operation actually learns medians, scales, categories and model coefficients: constructing the Python objects or calling `fit`?

In [3]:
numeric_pipe = Pipeline([
    ("fill", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("fill", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore")),
])
preprocess = ColumnTransformer([
    ("num", numeric_pipe, numeric),
    ("cat", categorical_pipe, categorical),
])
model = Pipeline([
    ("prep", preprocess),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])
print(model)
print("Example tunable parameter:", model.get_params()["model__C"])
print("No fitting has happened yet.")

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('fill',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scale',
                                                                   StandardScaler())]),
                                                  ['qualifying_position',
                                                   'practice_gap_s',
                                                   'previous_top10_rate',
                                                   'grid_penalty']),
                                                 ('cat',
                                                  Pipeline(steps=[('fill',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                  

## 3. Fit on training only, then inspect what was learned

This is the only `fit` call for the first candidate. The fitted preprocessing lives inside `model.named_steps["prep"]`. In particular, `statistics_` gives the four learned numeric medians in the order of `numeric`.

**Stop 3.** Predict whether a validation row can change those stored medians when we call `predict`.

In [4]:
model.fit(X_train, y_train)
learned = model.named_steps["prep"].named_transformers_["num"].named_steps["fill"].statistics_.copy()
print("Training medians:", dict(zip(numeric, learned.round(3))))
print("Transformed feature count:", len(model.named_steps["prep"].get_feature_names_out()))
print("First eight transformed names:", model.named_steps["prep"].get_feature_names_out()[:8].tolist())
assert len(learned) == len(numeric)
assert not any(pd.isna(learned))

Training medians: {'qualifying_position': np.float64(10.5), 'practice_gap_s': np.float64(0.695), 'previous_top10_rate': np.float64(0.493), 'grid_penalty': np.float64(0.0)}
Transformed feature count: 9
First eight transformed names: ['num__qualifying_position', 'num__practice_gap_s', 'num__previous_top10_rate', 'num__grid_penalty', 'cat__circuit_type_high_speed', 'cat__circuit_type_road', 'cat__circuit_type_street', 'cat__rain_forecast_no']


## 4. Validate the fitted procedure

A **baseline** is a fair simple comparison: it always predicts the most frequent class learned from training. We report accuracy and balanced accuracy because class proportions can make accuracy misleading. The validation period helps us compare approaches. The reserved test stays unopened.

`model.predict(X_valid)` performs the fitted preprocessing and classifier prediction in one call. We do not manually impute or encode validation.

**Stop 4.** Before running, identify the line that would be unsafe: `preprocess.fit_transform(data[features])` before the time split, or `model.predict(X_valid)` after fitting on training? Explain in one sentence.

In [5]:
baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
baseline_pred = baseline.predict(X_valid)
valid_pred = model.predict(X_valid)
print("Baseline accuracy / balanced accuracy:", round(accuracy_score(y_valid, baseline_pred),3), round(balanced_accuracy_score(y_valid, baseline_pred),3))
print("Pipeline accuracy / balanced accuracy:", round(accuracy_score(y_valid, valid_pred),3), round(balanced_accuracy_score(y_valid, valid_pred),3))
print("Stored medians unchanged after predict:", bool((learned == model.named_steps["prep"].named_transformers_["num"].named_steps["fill"].statistics_).all()))
assert (learned == model.named_steps["prep"].named_transformers_["num"].named_steps["fill"].statistics_).all()

Baseline accuracy / balanced accuracy: 0.5 0.5
Pipeline accuracy / balanced accuracy: 0.863 0.863
Stored medians unchanged after predict: True


## 5. The professional advantage: tune the whole procedure in temporal folds

Within the **training period only**, compare `model__C` values 0.1, 1 and 10. Two expanding folds are explicit: weeks 1–6 → 7–8, then weeks 1–8 → 9–10. In each fold, `GridSearchCV` clones the **entire pipeline** and fits its imputer, encoder, scaler and model again on that fold's earlier rows. A manual workflow can easily fit preprocessing once on all weeks 1–10 before cross-validation, leaking later-fold information.

The winning pipeline is refitted on weeks 1–10. We may then inspect weeks 11–14 once for this teaching comparison. We still do not open the reserved test.

**Stop 5.** In fold 1, which weeks may teach the imputer its medians? Where does `model__C` point?

In [6]:
folds = []
for last_train_week, first_val_week, last_val_week in [(6,7,8),(8,9,10)]:
    tr_idx = train.index[train.week <= last_train_week].to_numpy()
    va_idx = train.index[train.week.between(first_val_week,last_val_week)].to_numpy()
    folds.append((tr_idx, va_idx))
    print(f"Fold: train weeks 1–{last_train_week} ({len(tr_idx)} rows), validate {first_val_week}–{last_val_week} ({len(va_idx)} rows)")
search = GridSearchCV(model, {"model__C": [0.1, 1.0, 10.0]}, cv=folds, scoring="balanced_accuracy", refit=True)
search.fit(X_train, y_train)
print("Mean fold balanced accuracy by C:", [(p["model__C"],round(s,3)) for p,s in zip(search.cv_results_["params"],search.cv_results_["mean_test_score"])])
print("Selected C:", search.best_params_["model__C"])
print("Refitted pipeline validation balanced accuracy:", round(balanced_accuracy_score(y_valid, search.predict(X_valid)),3))
assert all(max(train.iloc[tr].week) < min(train.iloc[va].week) for tr,va in folds)

Fold: train weeks 1–6 (120 rows), validate 7–8 (40 rows)
Fold: train weeks 1–8 (160 rows), validate 9–10 (40 rows)
Mean fold balanced accuracy by C: [(0.1, np.float64(0.912)), (1.0, np.float64(0.888)), (10.0, np.float64(0.85))]
Selected C: 0.1
Refitted pipeline validation balanced accuracy: 0.875


## 6. Apply one fitted object to later rows, including an unseen category

`technical` appears later and was absent from training. With `handle_unknown="ignore"`, the encoder keeps the learned training columns and gives zeros for that category's one-hot columns; it does **not** learn a new category during prediction. A missing numeric input receives a stored training median. This is the same operation we would call when new rows arrive.

**Stop 6.** Change only the first row's `practice_gap_s` to missing in the copy below. Will the model's stored median change? Are individual predictions guaranteed to remain the same?

In [7]:
later = X_valid.iloc[[0,1,2]].copy()
later.loc[later.index[0], "circuit_type"] = "technical"
later.loc[later.index[0], "practice_gap_s"] = float("nan")
before = search.best_estimator_.named_steps["prep"].named_transformers_["num"].named_steps["fill"].statistics_.copy()
print("New-row features:", later.to_string(index=False))
print("Predicted top-ten labels:", search.predict(later).tolist())
print("Stored medians still the same:", bool((before == search.best_estimator_.named_steps["prep"].named_transformers_["num"].named_steps["fill"].statistics_).all()))
assert (before == search.best_estimator_.named_steps["prep"].named_transformers_["num"].named_steps["fill"].statistics_).all()

New-row features:

  qualifying_position  practice_gap_s  previous_top10_rate  grid_penalty circuit_type rain_forecast
                   6             NaN                0.264             0    technical            no
                  12           0.887                  NaN             0    technical            no
                  16           0.939                0.631             0    technical            no
Predicted top-ten labels: [1, 0, 0]
Stored medians still the same: True


## 7. Your change: edit one input, keep the fitted pipeline

The next cell starts with `new_qualifying_position = 20`. Before running, predict whether the top-ten label for the first later row will change. **Your pair's edit:** after the first run, set this number to **1**, rerun only this cell, and record the second outcome. Do **not** rerun `fit` or the parameter search. Explain why stored preprocessing values remain fixed even if a prediction changes.

In [8]:
new_qualifying_position = 20  # YOUR EDIT: after the first run, change 20 to 1 and rerun this cell.
scenario = later.iloc[[0]].copy()
scenario.loc[scenario.index[0], "qualifying_position"] = new_qualifying_position
print("Original qualifying position:", int(later.iloc[0].qualifying_position))
print("Scenario qualifying position:", new_qualifying_position)
print("Original / scenario predictions:", int(search.predict(later.iloc[[0]])[0]), int(search.predict(scenario)[0]))
print("Stored medians still the same:", bool((before == search.best_estimator_.named_steps["prep"].named_transformers_["num"].named_steps["fill"].statistics_).all()))

Original qualifying position: 6
Scenario qualifying position: 20
Original / scenario predictions: 1 0
Stored medians still the same: True


## 8. Diagnose the limits, then hand in

The one-line operation `search.predict(new_rows)` is useful because its preprocessing and model stay together. It does **not** select a valid time split, detect that a feature was measured after the race, or establish performance for every future circuit. `handle_unknown="ignore"` avoids an error for a new category; it does not prove good predictions on that category. The reserved test weeks 15–18 were not evaluated in this class.

**Pair discussion:** Which two checks must still be done by people? Give one possible leakage source outside the pipeline. Point to the object that contains the fitted median. Complete the worksheet and each person's exit ticket.

Reference: [scikit-learn Pipeline documentation](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

## L1. Create exactly one candidate feature

| Column | Role | Known when | Available after qualifying, before the race? |
|---|---|---|---|
| `qualifying_position` | source | end of qualifying | yes |
| `grid_penalty` | source | announced before the race start | yes (Week 4 Section 1) |
| `estimated_grid_position` | candidate feature | as soon as both sources are known | yes |
| `top10` | target | after the race | no, never an input |

The feature is computed **row by row** from that row's own pre-race values. It learns nothing from other rows (no mean, median or rank), so it cannot carry information between periods; the code checks this. Neither source column has missing values, so the feature needs no imputation of its own. We do not build it for the reserved test, because the test stays sealed.

*Why "estimated":* on a real grid, other drivers' penalties move a driver forward, and no grid slot is above 20. The formula ignores both, so a few rows get values above 20. We keep the formula from the lab brief; a within-weekend re-ranking would be a different feature.

In [9]:
FEATURE = "estimated_grid_position"
source_cols = ["qualifying_position", "grid_penalty"]

def add_estimated_grid_position(frame):
    """Pre-race feature: this row's qualifying slot plus its grid-place penalty."""
    out = frame.copy()
    out[FEATURE] = out["qualifying_position"] + out["grid_penalty"]
    return out

train_f = add_estimated_grid_position(train)
valid_f = add_estimated_grid_position(valid)
print("Source columns:", source_cols, "| both in the Week 4 pre-race inputs:", set(source_cols) <= set(numeric))
print("Missing source values (train / valid):", int(train_f[source_cols].isna().sum().sum()), int(valid_f[source_cols].isna().sum().sum()))
print("Rows with a grid penalty (train / valid):", int((train_f.grid_penalty > 0).sum()), "of", len(train_f), "/", int((valid_f.grid_penalty > 0).sum()), "of", len(valid_f))
print("Penalty values in training:", sorted(train_f.grid_penalty.unique().tolist()))
print("Rows with estimated_grid_position above 20 (train / valid):", int((train_f[FEATURE] > 20).sum()), int((valid_f[FEATURE] > 20).sum()))
pooled = add_estimated_grid_position(pd.concat([train, valid]))[FEATURE]
row_wise = bool(pooled.equals(pd.concat([train_f[FEATURE], valid_f[FEATURE]])))
print("Same values when computed per period or on weeks 1-14 together (no cross-row information):", row_wise)
print("Example training rows with a penalty:")
print(train_f.loc[train_f.grid_penalty > 0, ["week", "driver_id"] + source_cols + [FEATURE]].head(5).to_string(index=False))
assert set(source_cols) <= set(numeric) and "top10" not in source_cols + [FEATURE]
assert (train_f[FEATURE] == train_f.qualifying_position + train_f.grid_penalty).all()
assert row_wise and train_f.week.max() <= 10 and valid_f.week.between(11, 14).all()

Source columns: ['qualifying_position', 'grid_penalty'] | both in the Week 4 pre-race inputs: True
Missing source values (train / valid): 0 0
Rows with a grid penalty (train / valid): 22 of 200 / 9 of 80
Penalty values in training: [0, 3, 5]
Rows with estimated_grid_position above 20 (train / valid): 3 1
Same values when computed per period or on weeks 1-14 together (no cross-row information): True
Example training rows with a penalty:
 week driver_id  qualifying_position  grid_penalty  estimated_grid_position
    1       D05                    1             5                        6
    1       D12                   17             3                       20
    1       D14                   10             3                       13
    2       D14                    7             3                       10
    2       D19                   17             5                       22


## L2. Controlled comparison: same pipeline, same settings, ±1 feature

Both candidates use the **Section 2 pipeline**: median imputer + `StandardScaler` for numeric columns, most-frequent imputer + `OneHotEncoder(handle_unknown="ignore")` for categories, `LogisticRegression(C=1.0, max_iter=1000, random_state=42)`. The **only** difference is whether `estimated_grid_position` is in the numeric column list; the code prints every parameter that differs. We do not use the Section 5 search, because a separate search per candidate could pick a different `C` and add a second change.

Both candidates are fitted on **weeks 1–10 only** and scored once on the **same 80 rows of weeks 11–14**. The dummy row is the Week 4 `DummyClassifier(strategy="most_frequent")` from Section 4. The "without" candidate must reproduce the Section 4 predictions exactly, which ties the comparison to the Week 4 notebook.

In [10]:
from sklearn.base import clone

def build_pipeline(numeric_cols):
    """Fresh, unfitted copy of the Section 2 pipeline; only the numeric column list may change."""
    return Pipeline([
        ("prep", ColumnTransformer([
            ("num", clone(numeric_pipe), numeric_cols),
            ("cat", clone(categorical_pipe), categorical),
        ])),
        ("model", clone(model.named_steps["model"])),
    ])

numeric_plus = numeric + [FEATURE]
cols_without, cols_with = numeric + categorical, numeric_plus + categorical
without_feature = build_pipeline(numeric).fit(train_f[cols_without], y_train)
with_feature = build_pipeline(numeric_plus).fit(train_f[cols_with], y_train)

# Same settings check: every hyperparameter matches; only the column lists inside "prep" differ.
p0, p1 = without_feature.get_params(), with_feature.get_params()
differing = sorted(k for k in p0 if repr(p0[k]) != repr(p1[k]))
print("Parameters that differ:", differing)
assert set(differing) <= {"steps", "prep", "prep__transformers"}
assert p1["model__C"] == 1.0 and p1["model__max_iter"] == 1000 and p1["model__random_state"] == 42

# Prediction-time check: imputer medians come from weeks 1-10 only.
fill = with_feature.named_steps["prep"].named_transformers_["num"].named_steps["fill"]
train_only = bool((fill.statistics_ == train_f[numeric_plus].median().to_numpy()).all())
print(f"Imputer medians equal weeks 1–10 medians: {train_only} (fitted on {len(train_f)} training rows)")
assert train_only and len(train_f) == len(y_train) == 20 * 10

pred_without = without_feature.predict(valid_f[cols_without])
pred_with = with_feature.predict(valid_f[cols_with])
same_as_week4 = bool((pred_without == valid_pred).all())
print("Without-feature pipeline reproduces the Section 4 validation predictions:", same_as_week4)
assert same_as_week4

score_table = pd.DataFrame({
    "candidate": ["Dummy baseline (most frequent, Week 4)",
                  "Pipeline without feature",
                  "Pipeline with estimated_grid_position"],
    "model inputs": [f"none (always predicts {int(baseline_pred[0])})", "6 Week 4 columns", f"6 Week 4 columns + {FEATURE}"],
    "validation balanced accuracy (weeks 11–14)": [
        balanced_accuracy_score(y_valid, baseline_pred),
        balanced_accuracy_score(y_valid, pred_without),
        balanced_accuracy_score(y_valid, pred_with),
    ],
}).round(4)
print(f"Validation rows scored: {len(valid_f)} (weeks {valid_f.week.min()}–{valid_f.week.max()}; {int(y_valid.sum())} top-10, {int((y_valid == 0).sum())} not); reserved test rows scored: 0")
print(score_table.to_string(index=False))
print("Validation rows where the two pipelines disagree:", int((pred_without != pred_with).sum()), "of", len(valid_f))
assert len(score_table) == 3 and (valid_f.index == X_valid.index).all()

Parameters that differ: ['prep', 'prep__transformers', 'steps']
Imputer medians equal weeks 1–10 medians: True (fitted on 200 training rows)
Without-feature pipeline reproduces the Section 4 validation predictions: True
Validation rows scored: 80 (weeks 11–14; 40 top-10, 40 not); reserved test rows scored: 0
                             candidate                               model inputs  validation balanced accuracy (weeks 11–14)
Dummy baseline (most frequent, Week 4)                   none (always predicts 0)                                      0.5000
              Pipeline without feature                           6 Week 4 columns                                      0.8625
 Pipeline with estimated_grid_position 6 Week 4 columns + estimated_grid_position                                      0.8625
Validation rows where the two pipelines disagree: 0 of 80


## L3. Diagnostic: why the feature changes nothing (run after scoring)

The feature is an exact sum of two columns the model already receives. Logistic regression already fits a weighted sum of its inputs, so it could already weight a penalty place like a qualifying place without the new column; adding the sum gives it no new information and does not force that weighting. The code therefore asks: **how much does one grid-penalty place cost, compared with one qualifying place?** It converts the standardised coefficients to raw units (divides by the training standard deviation). The formula assumes a ratio of 1; a ratio near 0 means penalties barely matter. Training-row statistics use weeks 1–10 only.

In [11]:
def raw_unit_coefs(pipe, cols):
    """Log-odds change per raw unit: standardised coefficient / training standard deviation."""
    prep = pipe.named_steps["prep"]
    names = prep.get_feature_names_out()[:len(cols)].tolist()
    assert names == [f"num__{c}" for c in cols]
    return pd.Series(pipe.named_steps["model"].coef_[0][:len(cols)] / prep.named_transformers_["num"].named_steps["scale"].scale_, index=cols)

w0, w1 = raw_unit_coefs(without_feature, numeric), raw_unit_coefs(with_feature, numeric_plus)
ratio_without = w0.grid_penalty / w0.qualifying_position
ratio_with = (w1.grid_penalty + w1[FEATURE]) / (w1.qualifying_position + w1[FEATURE])
print("Training correlation of estimated_grid_position with qualifying_position:", round(train_f[FEATURE].corr(train_f.qualifying_position), 3))
print("Training top-10 rate, with / without a grid penalty:", round(train_f.loc[train_f.grid_penalty > 0, "top10"].mean(), 3), "/", round(train_f.loc[train_f.grid_penalty == 0, "top10"].mean(), 3))
print("Cost of 1 penalty place, in qualifying places (without / with feature):", round(ratio_without, 2), "/", round(ratio_with, 2))
prob_change = abs(with_feature.predict_proba(valid_f[cols_with])[:, 1] - without_feature.predict_proba(valid_f[cols_without])[:, 1])
print("Largest / mean change in a validation top-10 probability:", round(prob_change.max(), 3), "/", round(prob_change.mean(), 3))
print("Validation rows with a penalty:", int((valid_f.grid_penalty > 0).sum()), "of", len(valid_f), "| balanced accuracy moved by one changed prediction:", 0.5 / int(y_valid.sum()))

Training correlation of estimated_grid_position with qualifying_position: 0.98
Training top-10 rate, with / without a grid penalty: 0.5 / 0.5
Cost of 1 penalty place, in qualifying places (without / with feature): 0.31 / 0.32
Largest / mean change in a validation top-10 probability: 0.034 / 0.008
Validation rows with a penalty: 9 of 80 | balanced accuracy moved by one changed prediction: 0.0125


## L4. Decision (≤120 words)

**Reject `estimated_grid_position`.** Validation balanced accuracy on weeks 11–14 (L2): dummy baseline **0.5000**, without the feature **0.8625**, with the feature **0.8625**. Both pipelines predict the same label for all 80 rows. L0 expected no change or a very small increase; we observed no change. **Plausible reason:** the model already receives both source columns and learns their weights. Without the feature, one penalty place already costs about 0.31 qualifying places; with it, 0.32 (L3). In weeks 1–10, drivers with and without a penalty reached the top 10 equally often (0.50 vs 0.50). **Limitation:** only 80 validation rows from four synthetic weekends; one changed prediction moves the score by 0.0125. Real F1 performance cannot be inferred.

## L5. Brief AI recommendation audit

> *“The imputer does not use `top10`, so fit it once on all 18 weeks before splitting. This cannot leak information and gives a more accurate replacement for missing values.”*

1. **Reject.** Leakage means any information unavailable at prediction time, not only `top10`.
2. All-week medians would use validation and sealed-test inputs (weeks 11–18) that do not exist when week 11 is predicted, so validation would no longer imitate real use.
3. `In [2]` line 3 seals weeks 15–18 before any fit; `In [4]` line 1 `model.fit(X_train, y_train)` learns the printed `Training medians` from weeks 1–10.
4. `In [10]` output: `Imputer medians equal weeks 1–10 medians: True (fitted on 200 training rows)`.

**AI-use record**

- **Tool:** Claude (Anthropic) Opus 5.5, one chat on 8 Oct 2026, used by Vicente. We shared the lab brief, the Thursday notebook and the CSV. The AI did not execute code or see our results.
- **Representative prompt:** "Help me structure this laboratory and give me some pointers, then give me a congruent list to understand the objectives clearly."
- **Representative suggestion (paraphrased):** `estimated_grid_position` is a sum of two columns the logistic regression already uses, so expect little or no change; with 80 validation rows, one prediction = 0.0125 balanced accuracy.
- **Other suggestions:** notebook section order, fixing C for both models, listing the new column in the numeric features, which Thursday outputs to check for the audit, and commit/repo hygiene. It also drafted this record, which we edited.
- **Accepted:** The high-level proposal structure and diagnostic rationale (testing if logistic regression already subsumes the linear combination of `qualifying_position` and `grid_penalty`), as well as fixing hyperparameters ($C=1.0$) across both pipelines to guarantee a strictly controlled comparison.
- **Rejected or changed:** We rejected running a hyperparameter search (such as Section 5's `GridSearchCV`) to optimize $C$ separately per candidate, because varying $C$ would introduce a confounding factor rather than a pure ±1 feature comparison. We also wrote our own code, diagnostic implementations, and assertions rather than using AI-generated scripts.
- **Verification:** Our pipeline without the feature reproduced Thursday Section 4's validation score exactly (0.863 / 0.8625); we confirmed that both pipelines produced 0 disagreements across all 80 validation rows; and we verified the step size arithmetically ($0.5 / 40 = 0.0125$).
- **Limitation:** The AI's expectation about the feature was an analytical prediction made without running code or seeing empirical outputs; we implemented, executed, and verified all results independently. The code, numbers, proposal, decision, and audit text are ours.

In [12]:
import sys, platform, hashlib, numpy, sklearn
print("Python      ", sys.version.split()[0], "|", platform.platform())
print("numpy       ", numpy.__version__)
print("pandas      ", pd.__version__)
print("scikit-learn", sklearn.__version__)
# Line endings are normalised so a Windows checkout (CRLF) gives the same fingerprint.
print("Data file   ", path.name, "| rows:", len(data), "| SHA-256 (LF):", hashlib.sha256(path.read_bytes().replace(b"\r\n", b"\n")).hexdigest()[:16])

Python       3.12.15 | Linux-7.0.0-34-generic-x86_64-with-glibc2.43
numpy        2.1.2
pandas       2.2.3
scikit-learn 1.7.2
Data file    w04_race_weekends_synthetic_v3.csv | rows: 360 | SHA-256 (LF): b7dbafdcdffa9491


## L6. Package versions (above) and runbook

1. Keep `AIW_Lab02_G1_one_feature.ipynb` and `w04_race_weekends_synthetic_v3.csv` in the **same folder** (the repository root).
2. Create the environment with Python 3.12: `python3.12 -m venv .venv`, then `.venv/bin/pip install -r requirements.txt` (Windows: `py -3.12 -m venv .venv`, then `.venv\Scripts\pip install -r requirements.txt`).
3. Open the notebook with `jupyter notebook AIW_Lab02_G1_one_feature.ipynb` and choose **Kernel → Restart & Run All**. Headless alternative: `jupyter nbconvert --to notebook --execute --inplace AIW_Lab02_G1_one_feature.ipynb`.
4. Expected result: no assertion error; L2 table 0.5000 / 0.8625 / 0.8625 with 0 disagreeing rows; L3 penalty cost 0.31 / 0.32; data fingerprint as printed above. The run is deterministic (lbfgs solver, `random_state=42`, no random split). The same scores were obtained on Windows 10 and Linux.
5. Do **not** score weeks 15–18: this notebook never calls `predict` on `test_reserved`.